In [ ]:
# Breeze TTS 2 試聽：等零件的那三天，你在做什麼？（⚠️ 模型限研究／非商業用途，產生的聲音只做內部試聽）
# 右側 Settings：Accelerator 選 GPU（T4 x2 或 P100 都可，至少 12 GB），Internet 打開，然後 Run All。
# 模型如果要求先同意授權：到 huggingface.co/BreezeBlue/breeze-tts-2 按同意，
# 再把 Hugging Face token 加到 Kaggle 右側 Add-ons → Secrets，名稱 HF_TOKEN。
import os, subprocess, sys
os.chdir('/kaggle/working')
if not os.path.exists('breeze-tts'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/breezeblue-ai/breeze-tts.git'], check=True)
req = [l for l in open('breeze-tts/requirements.txt') if not l.startswith(('pytest', 'ruff'))]
open('req_kaggle.txt', 'w').writelines(req)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'req_kaggle.txt'], check=True)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
from huggingface_hub import snapshot_download
MODEL = snapshot_download('BreezeBlue/breeze-tts-2', local_dir='/kaggle/working/breeze-tts-2', token=token)
print('模型下載完成：', MODEL)


In [ ]:
# 這一集的台詞與角色聲音描述（由 匯出Kaggle筆記本_Breeze2.py 產生）
import json
LINES = json.loads("[{\"role\": \"主持人\", \"text\": \"零件沒到，客戶在催，你會怎麼回？\", \"out\": \"11_000\"}, {\"role\": \"主持人\", \"text\": \"最常聽到的是：「沒辦法，原廠還沒出貨。」\", \"out\": \"11_001\"}, {\"role\": \"主持人\", \"text\": \"這句話是真的，但客戶聽到的是：你們什麼都沒做。\", \"out\": \"11_002\"}, {\"role\": \"阿凱工程師\", \"text\": \"那次零件卡在海關三天。\", \"out\": \"11_003\"}, {\"role\": \"阿凱工程師\", \"text\": \"我想說也沒我能做的事，就等通知。\", \"out\": \"11_004\"}, {\"role\": \"阿凱工程師\", \"text\": \"結果客戶自己打去原廠問，還比我先知道進度。\", \"out\": \"11_005\"}, {\"role\": \"客戶\", \"text\": \"我還要自己去問原廠，那我找你們幹嘛？\", \"out\": \"11_006\"}, {\"role\": \"阿凱工程師\", \"text\": \"……\", \"out\": \"11_007\"}, {\"role\": \"主管\", \"text\": \"零件什麼時候到，你控制不了。\", \"out\": \"11_008\"}, {\"role\": \"主管\", \"text\": \"但每天回報進度、先做臨時處置、把裝機時間排好，這些你都控制得了。\", \"out\": \"11_009\"}, {\"role\": \"阿凱工程師\", \"text\": \"我以為沒消息就不用講。\", \"out\": \"11_010\"}, {\"role\": \"主管\", \"text\": \"沒消息，也是一種消息。\", \"out\": \"11_011\"}, {\"role\": \"主持人\", \"text\": \"這就是「主動積極」。\", \"out\": \"11_012\"}, {\"role\": \"主持人\", \"text\": \"把力氣放在你能影響的事，而不是只能擔心的事。\", \"out\": \"11_013\"}, {\"role\": \"主持人\", \"text\": \"所以，等待的時候可以做三件事。\", \"out\": \"11_014\"}, {\"role\": \"主持人\", \"text\": \"第一，就算沒進度，也固定時間主動回報。\", \"out\": \"11_015\"}, {\"role\": \"主持人\", \"text\": \"第二，先想有沒有臨時方案，讓客戶先撐過去。\", \"out\": \"11_016\"}, {\"role\": \"主持人\", \"text\": \"第三，問自己：這件事我還能多做哪一步？\", \"out\": \"11_017\"}, {\"role\": \"阿凱工程師\", \"text\": \"現在零件延誤，我會先打給客戶，不等他打給我。\", \"out\": \"11_018\"}, {\"role\": \"主持人\", \"text\": \"被動等，就是把主控權交給別人。\", \"out\": \"11_019\"}, {\"role\": \"主持人\", \"text\": \"你控制不了零件何時到，但你控制得了客戶何時知道。\", \"out\": \"11_quote\"}]")
ROLES = json.loads("{\"主持人\": {\"instruction\": \"一位三十歲左右的台灣女性節目主持人，說話快、直接、有梗，聲音明亮有自信，標準的台灣國語口音。\", \"seed\": 1000}, \"阿凱工程師\": {\"instruction\": \"一位三十歲左右的台灣男性工程師，聲音低沉一點，個性愛自嘲、有點無奈但很真誠，語速稍快，標準的台灣國語口音。\", \"seed\": 1001}, \"主管\": {\"instruction\": \"一位四十多歲的台灣女性主管，聲音沉穩有分量，說話不多但一句話戳重點，語速平穩，標準的台灣國語口音。\", \"seed\": 1002}, \"客戶\": {\"instruction\": \"一位著急的台灣女性客戶，說話短、直接、不太客氣，語速快，標準的台灣國語口音。\", \"seed\": 1003}}")
SAMPLE = "大家好，今天我們來聊聊在現場維修的時候，怎麼跟客戶溝通，讓事情處理得又快又好。"
print(len(LINES), '句，', len(ROLES), '個角色')


In [ ]:
# 載入模型（只載一次）
import os, sys, torch
from pathlib import Path
os.chdir('/kaggle/working/breeze-tts'); sys.path.insert(0, '/kaggle/working/breeze-tts')
import soundfile as sf
from breeze_infer.runtime import load_runtime, resolve_device, set_all_seeds, update_generation_config_for_breeze
from breeze_infer.templates import get_template, prepare_inputs, select_template_name
from models.fast_streaming import FastBreezeStreamingRuntime, FastStreamingConfig
print('GPU：', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
tok, model, atok = load_runtime(Path('/kaggle/working/breeze-tts-2'), device=resolve_device(), attn_implementation='eager')
update_generation_config_for_breeze(model)
runtime = FastBreezeStreamingRuntime(model, atok, FastStreamingConfig(max_new_tokens=1500, max_seq_len=2048, fast_all=False,
    fast_text_encoder=False, fast_backbone_prefill=False, fast_backbone_decode=False, fast_depth_decoder=False,
    fast_codec=False, repetition_penalty=1.1), tokenizer=tok)

def synth(text, out, seed, cfg=1.0, instruction=None, ref_audio=None, ref_text=None):
    req = {'id': 'r', 'text': text, 'speaker': 'S0'}
    if instruction: req['instruction'] = instruction
    if ref_audio: req['ref_audio_path'] = ref_audio; req['ref_text'] = ref_text
    set_all_seeds(seed)
    inputs = prepare_inputs(tok, atok, model, [req], get_template(select_template_name(req)),
                            guidance_scale=cfg, guidance_scale_ref=None, guidance_scale_ins=None)
    with sf.SoundFile(out, mode='w', samplerate=runtime.sample_rate, channels=1, subtype='PCM_16') as f:
        for chunk in runtime.iter_audio_chunks(inputs, request_id='r', seed=seed):
            f.write(chunk.audio)
print('模型載入完成')


In [ ]:
# 1) 每個角色用描述設計一段參考聲音　2) 用參考聲音複製來唸每一句（同一角色聲音才會一致）
os.makedirs('/kaggle/working/wav', exist_ok=True)
for role, r in ROLES.items():
    if not any(l['role'] == role for l in LINES): continue
    r['ref'] = f'/kaggle/working/wav/ref_{r["seed"]}.wav'
    synth(SAMPLE, r['ref'], r['seed'], cfg=4.0, instruction=r['instruction'])
    print('角色聲音：', role)
for n, l in enumerate(LINES):
    r = ROLES[l['role']]
    synth(l['text'], f"/kaggle/working/wav/{l['out']}.wav", 42 + n, ref_audio=r['ref'], ref_text=SAMPLE)
    print(n + 1, '/', len(LINES), l['text'][:20])


In [ ]:
# 轉 mp3 打包（ref_*.mp3 是各角色的參考聲音，也可以先聽聽看）
import glob, subprocess, zipfile
os.makedirs('/kaggle/working/mp3', exist_ok=True)
for w in sorted(glob.glob('/kaggle/working/wav/*.wav')):
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', w, '-ac', '1', '-b:a', '48k',
                    '/kaggle/working/mp3/' + os.path.basename(w)[:-4] + '.mp3'], check=True)
zp = '/kaggle/working/11_語音_breeze2.zip'
with zipfile.ZipFile(zp, 'w') as z:
    for m in sorted(glob.glob('/kaggle/working/mp3/*.mp3')):
        z.write(m, os.path.basename(m))
print('完成 →', zp)
